In [ ]:
from langgraph.graph import StateGraph,START, END # type: ignore
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI 
from dotenv import load_dotenv
from typing import TypedDict

c:\Users\dell\Desktop\Langgraph-learning\agent\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.7) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\dell\Desktop\Langgraph-learning\agent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [ ]:
# defining sub graph for translation
class substate(TypedDict):
    input : str
    translated : str 

def translate_to_french(state: substate):
    input = state['input']
    prompt = PromptTemplate(
        input_variables=["input"],
        template="Translate the following English text to French: {input}"
    )
    response = llm.invoke(prompt.format(input=input)).content
    return {"translated" : response}

In [5]:
graph = StateGraph(substate)
graph.add_node("translate", translate_to_french)
graph.add_edge(START,"translate")
graph.add_edge("translate", END)
Subgraph = graph.compile()

In [8]:
# defining main graph
class mainstate(TypedDict):
    query : str
    answer : str 
    translated_ans : str

def answer_query(state: mainstate):
    query = state['query']
    prompt = PromptTemplate(
        input_variables=["query"],
        template="Answer the following question: {query}"
    )
    response = llm.invoke(prompt.format(query=query)).content
    return {"answer" : response}

def translated_answer(state : mainstate):
    result = Subgraph.invoke({"input" : state['answer']})

    return {"translated_ans" : result['translated']}
    

In [9]:
graph = StateGraph(mainstate)
graph.add_node("answer_query", answer_query)
graph.add_node("translate_answer", translated_answer)
graph.add_edge(START,"answer_query")
graph.add_edge("answer_query", "translate_answer")
graph.add_edge("translate_answer", END)

MainGraph = graph.compile()

In [11]:
MainGraph.invoke({"query" : "what is full form of IVM ?"})

{'query': 'what is full form of IVM ?',
 'answer': 'The most common full form for **IVM** is:\n\n*   **In Vitro Maturation** (a fertility treatment and reproductive technology)\n\nHowever, IVM can also stand for other things depending on the context, such as:\n\n*   **Integrated Vendor Management**\n*   **Intelligent Vehicle Management**\n*   **Individualized Vocational Model**\n*   **Integrated Virtual Machine**\n*   **Information Volume Management**\n\nIf you have a specific field or context in mind, I might be able to give you a more precise answer!',
 'translated_ans': "La forme complète la plus courante pour **IVM** est :\n\n*   **Maturation in vitro** (un traitement de fertilité et une technologie de reproduction)\n\nCependant, IVM peut également signifier d'autres choses selon le contexte, tels que :\n\n*   **Gestion intégrée des fournisseurs** (Integrated Vendor Management)\n*   **Gestion intelligente des véhicules** (Intelligent Vehicle Management)\n*   **Modèle professionnel 